# babaorum DATA-1 — fragments B3 et S2 pour pR, morceau 2 sur 3 (Kaggle CPU, données seulement)
Lot : docs/END_TO_END_DATA1_LOT.txt (amendements 1 et 2). Aucune métrique, aucune vérité, aucun GPU, pas de fusion. Entrées : (1) bundle v5 (sha ca41c092) ; (2) kit E2E-4 et fichiers DATA-1 ; (3) sortie E2E-1 (calibration) ; (4) sortie E2E-4 v3 (e2e4_extraction_report.json, référence des sha) ; (5) sortie du carnet babaorum-e2e3-run3-fp-classes (e2e3/mf). Sorties : /kaggle/working/fragments/<bras>/*.json du morceau 2, data1_manifest.json. Seuls le statut et les temps sont affichés.

In [ ]:
import importlib, os, subprocess, sys
REQUIRED_OUTSIDE_IMAGE = []          # paquets pip hors image ; vide pour la chaîne d'E2E-5
for pkg in REQUIRED_OUTSIDE_IMAGE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
for mod in ('numpy', 'torch', 'PIL', 'lxml', *REQUIRED_OUTSIDE_IMAGE):
    importlib.import_module(mod)     # échoue tout de suite si l'image ne suffit pas


In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

BUNDLE_SHA = "ca41c09272248c5096f2547fe738eee622db5af50414182537ac8e94e43367f5"
KIT_SHA = {"scripts/e2e3_fp_classes.py": "fc96238085337b0cae2303fb717671a32ec0758048f9b252dd54c1e3a32f299b", "scripts/g_diag_mf.py": "aedd499722504468cb8c1764e174ae7137830b33b2043f3c8f47464d885d6f0a", "scripts/e2e4_extraction.py": "e874753604d7772709d13cbd997f8dcc61a5aad97cdf96055f908ccd5a24b386", "config/e2e4_extraction.json": "a2693cbc3447f5e66f76d3882cea3710b73dde6b08ff1ab15faac0cf464596e4", "scripts/data1_fragments.py": "722c2262c1dd36b246f9084d2e3282c322f90e2f64ef0d2d4d3b120efc3db779", "config/data1_fragments.json": "84d4b11154994dfc6ecb8f3d713e9780f3a3b5a0aadd91f04bd63da2f5971866", "tests/test_data1_fragments.py": "6466295db05fe941621fe3a189edd149696b35ebad2760a81e8b0540509bca27"}
SHARD = "2/3"
HARD_STOP_S = 510 * 60
OUT = Path("/kaggle/working")


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) == 1, (pattern, hits)
    return Path(hits[0])


# 0. cheap controls: ps, torch and numpy versions (CPU run)
import torch  # noqa: E402
import numpy  # noqa: E402
assert shutil.which("ps"), "ps missing"
print("torch", torch.__version__, "numpy", numpy.__version__, "shard", SHARD, flush=True)

# 1. bundle v5, checked then extracted (the engine of E2E-4, unchanged)
zip_path = one("/kaggle/input/**/stage_s_bundle_v5.zip.bin")  # .bin: Kaggle unzips .zip uploads
got = sha256(zip_path)
assert got == BUNDLE_SHA, ("bundle sha mismatch", got)
BUNDLE = OUT / "bundle"
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True)
zipfile.ZipFile(zip_path).extractall(BUNDLE)

# 2. DATA-1 kit overlaid on the babaorum tree of the bundle; E2E-4 files must equal the frozen E2E-4 shas
kit_script = one("/kaggle/input/**/data1_fragments.py")
KIT = kit_script.parent.parent
kit_sha = {}
for rel, want in KIT_SHA.items():
    src = KIT / rel
    assert src.exists(), ("kit file missing", rel)
    kit_sha[rel] = sha256(src)
    assert kit_sha[rel] == want, ("kit sha mismatch", rel, kit_sha[rel])
    dst = BUNDLE / "babaorum" / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
ROOT_D1 = BUNDLE / "babaorum"
PYTHONPATH = os.pathsep.join([str(ROOT_D1 / "src"), str(ROOT_D1 / "scripts")])

# 3. E2E-1 checkpoints (folder containing B3/ and S2/), E2E-4 v3 report (page sha reference), E2E-3 run 3 pages
ckpt = {arm: one(f"/kaggle/input/**/calibration/{arm}/checkpoint_last.pt") for arm in ("B3", "S2")}
assert sha256(ckpt["B3"]).startswith("738ca116bdfcef16") and sha256(ckpt["S2"]).startswith("12fca335")
CALIB = ckpt["B3"].parent.parent
assert CALIB == ckpt["S2"].parent.parent, "checkpoints from different folders"
E4_REPORT = one("/kaggle/input/**/e2e4_extraction_report.json")
assert sha256(E4_REPORT).startswith("714bc571"), "E2E-4 report sha differs from 714bc571"
mf_dirs = [Path(p) for p in glob.glob("/kaggle/input/**/e2e3/mf", recursive=True) if Path(p).is_dir()]
assert len(mf_dirs) == 1, ("e2e3/mf not found once", mf_dirs)
PAGES_TMP = mf_dirs[0].parent

# 4. tests of the DATA-1 rules (cheap, before the run); pytest is in the Kaggle image
env = dict(os.environ, PYTHONPATH=PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_data1_fragments.py",
                "--deselect", "tests/test_data1_fragments.py::test_install_cell_of_the_builder_is_the_block_of_kaggle_install_txt",
                "--deselect", "tests/test_data1_fragments.py::test_three_notebooks_each_with_its_shard_and_the_install_cell_first"],
               cwd=str(ROOT_D1), env=env, check=True, text=True)

# 5. manifest, then the shard run with a hard stop (outputs only in /kaggle/working)
manifest = {"bundle_sha256": got, "kit_sha256": kit_sha, "e2e4_report_sha256": sha256(E4_REPORT),
            "shard": SHARD, "python": platform.python_version(),
            "torch": version("torch"), "numpy": version("numpy")}
print(json.dumps(manifest, indent=2), flush=True)
try:
    proc = subprocess.run([sys.executable, "scripts/data1_fragments.py", "--confirmed-go",
                           "--tmp", str(PAGES_TMP), "--calibration", str(CALIB),
                           "--e2e4-report", str(E4_REPORT), "--shard", SHARD, "--out-dir", str(OUT)],
                          cwd=str(ROOT_D1), env=env, capture_output=True, text=True, timeout=HARD_STOP_S)
    print(proc.stdout.strip(), flush=True)
    rc = proc.returncode
except subprocess.TimeoutExpired:
    print(json.dumps({"status": "INCOMPLETE", "reason": "hard stop 510 min"}), flush=True)
    rc = "timeout"
(OUT / "data1_run_exit.json").write_text(json.dumps({"exit_code": rc}))
